In [37]:
import pandas as pd
fund = pd.read_csv("/Users/chitranjansahu/Desktop/mutual_fund_analysis/data/raw/01_fund_master.csv")
nav = pd.read_csv("/Users/chitranjansahu/Desktop/mutual_fund_analysis/data/raw/02_nav_history.csv")  
tx  = pd.read_csv("/Users/chitranjansahu/Desktop/mutual_fund_analysis/data/raw/08_investor_transactions.csv")
perf = pd.read_csv("/Users/chitranjansahu/Desktop/mutual_fund_analysis/data/raw/07_scheme_performance.csv")
aum = pd.read_csv("/Users/chitranjansahu/Desktop/mutual_fund_analysis/data/raw/03_aum_by_fund_house.csv")
nav["date"] = pd.to_datetime(nav["date"])

In [13]:
nav = nav.sort_values(
    ["amfi_code","date"]
)

#nav>0
nav.drop_duplicates(inplace=True)
nav = nav[nav["nav"]>0]

#forward fill
nav["nav"] = (
    nav.groupby("amfi_code")["nav"]
       .ffill()
)

clean Transactions

In [14]:
tx["transaction_type"] = (
    tx["transaction_type"]
      .str.upper()
)

Mapping 

In [15]:
mapping = {
    "sip":"SIP",
    "lumpsum":"Lumpsum",
    "redemption":"Redemption"
}

In [19]:
tx = tx[tx["amount_inr"]>0]

Clean Performance 

In [26]:
cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct"
]

for c in cols:
    perf[c] = pd.to_numeric(
        perf[c],
        errors="coerce"
    )

In [28]:
perf[
    (perf["expense_ratio_pct"]<0.1)
    |
    (perf["expense_ratio_pct"]>2.5)
]

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade


In [30]:
tx.to_csv(
    "/Users/chitranjansahu/Desktop/mutual_fund_analysis/data/processed/transactions_cleaned.csv",
    index=False
)

In [31]:
perf.to_csv(
    "/Users/chitranjansahu/Desktop/mutual_fund_analysis/data/processed/scheme_performance_cleaned.csv",
    index=False
)

In [32]:
from sqlalchemy import create_engine

engine = create_engine(
    "sqlite:///../data/db/bluestock_mf.db"
)

In [38]:
fund.to_sql(
    "dim_fund",
    engine,
    if_exists="replace",
    index=False
)

nav.to_sql(
    "fact_nav",
    engine,
    if_exists="replace",
    index=False
)

tx.to_sql(
    "fact_transactions",
    engine,
    if_exists="replace",
    index=False
)

perf.to_sql(
    "fact_performance",
    engine,
    if_exists="replace",
    index=False
)

aum.to_sql(
    "fact_aum",
    engine,
    if_exists="replace",
    index=False
)

90

In [42]:
query = """
SELECT *
FROM fact_aum
ORDER BY aum_crore DESC
LIMIT 5;
"""

result = pd.read_sql(query, engine)
result

,date,fund_house,aum_lakh_crore,aum_crore,num_schemes,Unnamed: 5,Unnamed: 6
0,2025-03-31,SBI Mutual Fund,12.50,1250000,186,None,None
1,2025-12-31,SBI Mutual Fund,12.50,1250000,186,None,None
2,2024-12-31,SBI Mutual Fund,11.14,1114000,186,None,None
3,2024-09-30,SBI Mutual Fund,10.80,1080000,186,None,None
4,2025-12-31,ICICI Prudential MF,10.74,1074000,216,None,None


In [40]:
pd.read_sql(
    "PRAGMA table_info(fact_aum);",
    engine
)

,cid,name,type,notnull,dflt_value,pk
0,0,date,TEXT,0,None,0
1,1,fund_house,TEXT,0,None,0
2,2,aum_lakh_crore,FLOAT,0,None,0
3,3,aum_crore,BIGINT,0,None,0
4,4,num_schemes,BIGINT,0,None,0
5,5,Unnamed: 5,FLOAT,0,None,0
6,6,Unnamed: 6,FLOAT,0,None,0


In [41]:
fact_aum = pd.read_sql(
    "SELECT * FROM fact_aum LIMIT 5;",
    engine
)

fact_aum.columns

Index(['date', 'fund_house', 'aum_lakh_crore', 'aum_crore', 'num_schemes',
       'Unnamed: 5', 'Unnamed: 6'],
      dtype='object')

In [44]:
print(fund.dtypes)

amfi_code               int64
fund_house             object
scheme_name            object
category               object
sub_category           object
plan                   object
launch_date            object
benchmark              object
expense_ratio_pct     float64
exit_load_pct         float64
min_sip_amount          int64
min_lumpsum_amount      int64
fund_manager           object
risk_category          object
sebi_category_code     object
dtype: object
